<a href="https://colab.research.google.com/github/LoneWolf2026/Neural-Network-for-Nuclear-Binding-Energy-Prediction-/blob/main/Nuclear_Binding_Energy_NeuralNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Omit uncertainties for now. Add them later. Focus on getting the Neural net trained first
#1. Optimize Gradient Descent functions
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [ ]:
data_df_2020 = pd.read_csv('/content/drive/MyDrive/AME_Dataset_2020.csv',header =None)
data_df_2020 = data_df_2020.drop([0, 1]).reset_index(drop=True) #removes neutron and proton from first two rows
data_df_2020 = data_df_2020.drop(columns=[0,4]).reset_index(drop=True)
data_df_2020.columns = range(data_df_2020.columns.size)
data_df_2020.head()

,0,1,2,3,4,5,6,7,8
0,1,1,2,13135.722895,0.000015,1112.28310,0.00020,14101.777844,0.000015
1,2,1,3,14949.810900,0.000080,2827.26540,0.00030,16049.281320,0.000080
2,1,2,3,14931.218880,0.000060,2572.68044,0.00015,16029.321970,0.000060
3,0,3,3,28667.000000,2000.000000,-2267.00000,667.00000,30775.000000,2147.000000
4,3,1,4,24621.129000,100.000000,1720.44910,25.00000,26431.867000,107.354000


In [ ]:
og_data = data_df_2020.copy()


In [ ]:
input = np.array(data_df_2020.iloc[:,[0,1,2,3,7]].values) #Neutrons, Protons, Mass Number, Mass Excess, and Atomic Mass are take as inputs
output = np.array(data_df_2020.iloc[:,[5]].values) #Binding Energy is our output


In [ ]:
input_train,input_test,output_train,output_test = train_test_split(input,output,test_size = 0.2)

#Standardize input and output training data for simpler training
scaler_input = StandardScaler()
scaler_output = StandardScaler()
#must have two different scaler functions since input and output have different column dimensions

input_train = scaler_input.fit_transform(input_train)
output_train = scaler_output.fit_transform(output_train)

input_test = scaler_input.transform(input_test)
output_test = scaler_output.transform(output_test)

In [ ]:
class dataset(Dataset):
  def __init__(self,input,output):
    self.input = torch.tensor(input, dtype = torch.float32).to(device)
    self.output = torch.tensor(output, dtype = torch.float32).to(device)

  def __len__(self):
    return len(self.input)

  def __getitem__(self,index):
    return self.input[index], self.output[index]

In [ ]:
training_data = dataset(input_train,output_train)
testing_data = dataset(input_test,output_test)

In [ ]:
train_dataloader = DataLoader(training_data,batch_size = 64,shuffle = True)
test_dataloader = DataLoader(testing_data,batch_size = 64,shuffle = True)

In [ ]:

class BindingE_NeuralNet(nn.Module):
  def __init__(self):
    super(BindingE_NeuralNet,self).__init__()

    self.layer1 = nn.Linear(input_train.shape[1],64)
    self.layer2 = nn.Linear(64,32)
    self.OutLayer = nn.Linear(32,1)
    self.Relu = nn.ReLU()

  def forward(self,X):

    X = self.Relu(self.layer1(X))
    X = self.Relu(self.layer2(X))
    X = self.OutLayer(X)

    return X


BE_NeuralNet = BindingE_NeuralNet().to('cuda')

In [ ]:
summary(BE_NeuralNet,(input.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 64]             384
              ReLU-2                   [-1, 64]               0
            Linear-3                   [-1, 32]           2,080
              ReLU-4                   [-1, 32]               0
            Linear-5                    [-1, 1]              33
Total params: 2,497
Trainable params: 2,497
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.01
Estimated Total Size (MB): 0.01
----------------------------------------------------------------


In [ ]:
criterion = nn.MSELoss() #Loss function, uses Mean Squared Error
optimizer = Adam(BE_NeuralNet.parameters(), lr = 0.001) #Gradient Descent algorithm (Adam)

In [ ]:
total_loss_train_plot = []
total_loss_test_plot = []
total_acc_train_plot = []
total_acc_test_plot = []

avg_loss = [] #Avg loss is defined as testing loss

epochs = 20
for epoch in range(epochs):
  total_loss_train = 0
  total_acc_test = []
  total_loss_test = 0

  for data in train_dataloader:
    inputs, output = data

    prediction = BE_NeuralNet(inputs) #Prediction by neural network

    batch_loss = criterion(prediction,output) #calculate loss for current batch
    total_loss_train += batch_loss.item()

    batch_loss.backward() #Backpropagation done on each batch of data
    optimizer.step() #Adjust weights of parameters
    optimizer.zero_grad() #Reset gradient for next batch

  avg_loss_train = total_loss_train/len(train_dataloader) #Average training data loss (before backpropagation) over current epoch

  with torch.no_grad(): #Stop tracking gradients of test data (as it is not needed)
    for data in test_dataloader:
      inputs, output = data

      prediction = BE_NeuralNet(inputs)

      batch_loss = criterion(prediction,output)
      total_loss_test += batch_loss.item()

    avg_test_loss = total_loss_test/len(test_dataloader) #Average testing data loss (before backpropagation) over current epoch

    #Accuracy is not a strong metric for this particular neural network, which is predicting decimal values. Therefore, only the loss function is displayed.

    avg_loss.append(avg_test_loss) #Avg loss is defined as testing loss


  print(f'''Epoch: {epoch+1} Train_Loss: {avg_loss_train}
        Testing Loss: {avg_loss_test}''')
  print("=="*25)

avg_loss = sum(avg_loss)/len(avg_loss)
print(f"Avg Loss Final: {avg_loss}")

Epoch: 1 Train_Loss: 0.6659614594446288
        Testing Loss: 0.16540675703436136
Epoch: 2 Train_Loss: 0.35723806901110544
        Testing Loss: 0.10510733226935069
Epoch: 3 Train_Loss: 0.3286598823550675
        Testing Loss: 0.0935410219244659
Epoch: 4 Train_Loss: 0.3169373076202141
        Testing Loss: 0.22948294846961895
Epoch: 5 Train_Loss: 0.3002504363552564
        Testing Loss: 0.08770503057166934
Epoch: 6 Train_Loss: 0.2846927131733133
        Testing Loss: 0.08181724200646083
Epoch: 7 Train_Loss: 0.2714701778151923
        Testing Loss: 0.07039419394762565
Epoch: 8 Train_Loss: 0.28577505143152343
        Testing Loss: 0.05418481743739297
Epoch: 9 Train_Loss: 0.2459831419814792
        Testing Loss: 0.04613903175534991
Epoch: 10 Train_Loss: 0.23805935559794306
        Testing Loss: 0.08553699140126507
Epoch: 11 Train_Loss: 0.22859101843916707
        Testing Loss: 0.056290452756608524
Epoch: 12 Train_Loss: 0.20747451394692892
        Testing Loss: 0.0355186587330536
Epoch: 13

In [ ]:
test_data_num2 = dataset(input_test,output_test)
test_dataloader_num2 = DataLoader(test_data_num2,batch_size = 8,shuffle = True)
NeuralNet_predicts = []

for data in test_dataloader_num2:
  input = data

  prediction = BE_NeuralNet(input)
  NeuralNet_predicts += prediction


x1 = input_test[:,2]
y1 = NeuralNet_predicts
plt.plot(x1,y1)
plt.show()

#A second set of testing data is used as a final measure of the Neural Networks' quality. At time of writing the same data set is recycled, but I will consider using a different data set. Additionally, I may remove this second set of testing data all together and move the testing data used during training to be used post-training.